In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter  # Import TensorBoard
from torchvision import transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
import os
from medmnist import PathMNIST
from tqdm import tqdm
import torch.nn.functional as F
from torchvision.models import inception_v3
from torchvision.transforms import functional as TF
from scipy import linalg

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
num_epochs = 100
batch_size = 128
lr_d = 0.0001
lr_g = 0.0001
z_dim = 100
n_critic = 5
lambda_gp = 10

# Data loading and preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = PathMNIST(split='train', transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

Using downloaded and verified file: C:\Users\Sudhanshu\.medmnist\pathmnist.npz


In [3]:
# Generator for WGAN-GP
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 3, 1, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# Critic for WGAN-GP
class Critic(nn.Module):
    def __init__(self):
        super(Critic, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 1, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.model(x)

In [4]:
# Function to compute gradient penalty
def compute_gradient_penalty(critic, real_samples, fake_samples, device):
    batch_size = real_samples.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    critic_interpolates = critic(interpolates)
    grad_outputs = torch.ones_like(critic_interpolates, device=device)
    gradients = torch.autograd.grad(
        outputs=critic_interpolates,
        inputs=interpolates,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    gradients = gradients.view(batch_size, -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

In [5]:
# Visualization function
def visualize(real_images, generated_images, epoch, model_name):
    real_images = (real_images + 1) / 2
    generated_images = (generated_images + 1) / 2
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for i in range(5):
        axes[0, i].imshow(real_images[i].permute(1, 2, 0).cpu())
        axes[0, i].axis('off')
        axes[1, i].imshow(generated_images[i].permute(1, 2, 0).cpu())
        axes[1, i].axis('off')
    plt.suptitle(f'{model_name} - Epoch {epoch+1}')
    plt.savefig(f'generated_images_{model_name}/epoch_{epoch+1}_comparison.png')
    plt.close()

In [6]:
# Function to load Inception V3 model
def load_inception_model(device):
    inception_model = inception_v3(weights='Inception_V3_Weights.IMAGENET1K_V1').to(device)
    inception_model.eval()
    return inception_model

# Function to preprocess images for Inception V3
def preprocess_for_inception(images, device):
    images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
    return images

# Function to compute activations from Inception V3
def get_inception_activations(images, inception_model, device, batch_size=32):
    images = preprocess_for_inception(images, device)
    activations = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size].to(device)
            act = inception_model(batch)
            activations.append(act.cpu().numpy())
    return np.concatenate(activations, axis=0)

# Function to compute FID score
def compute_fid(real_images, fake_images, inception_model, device):
    real_acts = get_inception_activations(real_images, inception_model, device)
    fake_acts = get_inception_activations(fake_images, inception_model, device)
    
    mu_real, sigma_real = np.mean(real_acts, axis=0), np.cov(real_acts, rowvar=False)
    mu_fake, sigma_fake = np.mean(fake_acts, axis=0), np.cov(fake_acts, rowvar=False)
    
    diff = mu_real - mu_fake
    covmean = linalg.sqrtm(sigma_real.dot(sigma_fake), disp=False)[0]
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

# Function to compute Inception Score
def compute_inception_score(images, inception_model, device, splits=10, batch_size=32):
    images = preprocess_for_inception(images, device)
    preds = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size].to(device)
            pred = inception_model(batch)
            pred = F.softmax(pred, dim=1).cpu().numpy()
            preds.append(pred)
    preds = np.concatenate(preds, axis=0)
    
    scores = []
    for i in range(splits):
        part = preds[(i * preds.shape[0] // splits):((i + 1) * preds.shape[0] // splits)]
        kl = part * (np.log(part) - np.log(np.mean(part, axis=0, keepdims=True)))
        kl = np.mean(np.sum(kl, axis=1))
        scores.append(np.exp(kl))
    return np.mean(scores), np.std(scores)

# Function to evaluate using FID and IS
def evaluate(generator, train_loader, device, num_samples=5000, z_dim=100):
    inception_model = load_inception_model(device)
    
    # Generate fake images
    generator.eval()
    fake_images = []
    with torch.no_grad():
        for _ in range(num_samples // 128):
            z = torch.randn(128, z_dim, 1, 1).to(device)
            fake = generator(z)
            fake_images.append(fake.cpu())
    fake_images = torch.cat(fake_images, dim=0)[:num_samples]
    
    # Sample real images
    real_images = []
    for batch in train_loader:
        real = batch[0]
        real_images.append(real)
        if len(real_images) * 128 >= num_samples:
            break
    real_images = torch.cat(real_images, dim=0)[:num_samples]
    
    # Compute FID
    fid_score = compute_fid(real_images, fake_images, inception_model, device)
    
    # Compute Inception Score
    is_mean, is_std = compute_inception_score(fake_images, inception_model, device)
    
    return fid_score, is_mean, is_std

In [7]:
# Training function for WGAN-GP with TensorBoard logging
def train_wgan_gp():
    # Initialize models
    generator = Generator(z_dim=z_dim).to(device)
    critic = Critic().to(device)
    
    # Optimizers
    optimizer_c = optim.Adam(critic.parameters(), lr=lr_d, betas=(0.0, 0.9))
    optimizer_g = optim.Adam(generator.parameters(), lr=lr_g, betas=(0.0, 0.9))
    
    # Initialize TensorBoard writer
    writer = SummaryWriter('runs/wgan_gp_experiment')
    
    # Create directory for saving images (optional, since we'll use TensorBoard)
    os.makedirs('generated_images_WGAN_GP', exist_ok=True)
    
    # Best model tracking
    best_c_loss = float('inf')
    
    # Fixed noise for consistent visualization
    fixed_z = torch.randn(5, z_dim, 1, 1).to(device)
    
    # Training loop
    for epoch in range(num_epochs):
        critic.train()
        generator.train()
        c_loss_total = 0.0
        g_loss_total = 0.0
        
        for i, (real_images, _) in enumerate(tqdm(train_loader, desc=f"WGAN-GP Epoch {epoch+1}/{num_epochs}")):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)
            
            # Train Critic
            for _ in range(n_critic):
                optimizer_c.zero_grad()
                
                # Add noise to real images
                noise = torch.randn_like(real_images) * 0.05
                real_images_noisy = real_images + noise
                real_output = critic(real_images_noisy)
                c_loss_real = -real_output.mean()
                
                # Fake images for critic
                z = torch.randn(batch_size, z_dim, 1, 1).to(device)
                fake_images_critic = generator(z)
                fake_output = critic(fake_images_critic.detach())
                c_loss_fake = fake_output.mean()
                
                # Gradient penalty
                gp = compute_gradient_penalty(critic, real_images, fake_images_critic, device)
                c_loss = c_loss_real + c_loss_fake + lambda_gp * gp
                
                c_loss.backward()
                optimizer_c.step()
            
            c_loss_total += c_loss.item()
            
            # Train Generator
            optimizer_g.zero_grad()
            z = torch.randn(batch_size, z_dim, 1, 1).to(device)
            fake_images_gen = generator(z)
            fake_output = critic(fake_images_gen)
            g_loss = -fake_output.mean()
            g_loss.backward()
            optimizer_g.step()
            
            g_loss_total += g_loss.item()
            
            # Compute c_wasserstein for logging
            c_wasserstein = (real_output.mean() - fake_output.mean()).item()
        
        # Average losses for the epoch
        c_loss_avg = c_loss_total / len(train_loader)
        g_loss_avg = g_loss_total / len(train_loader)
        
        # Log losses and Wasserstein distance to TensorBoard
        writer.add_scalar('Loss/Critic', c_loss_avg, epoch)
        writer.add_scalar('Loss/Generator', g_loss_avg, epoch)
        writer.add_scalar('Metrics/Wasserstein Distance', c_wasserstein, epoch)
        
        print(f"WGAN-GP Epoch {epoch+1}: c_loss={c_loss_avg:.4f}, g_loss={g_loss_avg:.4f}, c_wasserstein={c_wasserstein:.4f}")
        
        # Save best model based on c_loss
        if c_loss_avg < best_c_loss:
            best_c_loss = c_loss_avg
            torch.save(generator.state_dict(), 'best_generator_wgan_gp.pth')
            print(f"Saved best WGAN-GP generator at epoch {epoch+1}")
        
        # Visualize and log images to TensorBoard every 5 epochs
        if (epoch + 1) % 5 == 0:
            with torch.no_grad():
                fake_images = generator(fixed_z)
                # Denormalize images for visualization
                real_images_vis = (real_images[:5] + 1) / 2
                fake_images_vis = (fake_images + 1) / 2
                # Create a grid of real and fake images
                real_grid = vutils.make_grid(real_images_vis, nrow=5, normalize=False)
                fake_grid = vutils.make_grid(fake_images_vis, nrow=5, normalize=False)
                # Log to TensorBoard
                writer.add_image('Images/Real', real_grid, epoch)
                writer.add_image('Images/Generated', fake_grid, epoch)
    
    # Final evaluation after training
    print("Training completed. Performing final evaluation...")
    fid_score, is_mean, is_std = evaluate(generator, train_loader, device, num_samples=5000, z_dim=z_dim)
    print(f"Final FID Score: {fid_score:.2f}")
    print(f"Final Inception Score: {is_mean:.2f} ± {is_std:.2f}")
    
    # Log final metrics to TensorBoard
    writer.add_scalar('Metrics/FID', fid_score, num_epochs)
    writer.add_scalar('Metrics/Inception Score Mean', is_mean, num_epochs)
    writer.add_scalar('Metrics/Inception Score Std', is_std, num_epochs)
    
    # Close the TensorBoard writer
    writer.close()

# Run the training
if __name__ == "__main__":
    train_wgan_gp()

WGAN-GP Epoch 1/100: 100%|██████████| 703/703 [01:29<00:00,  7.83it/s]


WGAN-GP Epoch 1: c_loss=-9.0450, g_loss=11.0562, c_wasserstein=-0.7378
Saved best WGAN-GP generator at epoch 1


WGAN-GP Epoch 2/100: 100%|██████████| 703/703 [01:28<00:00,  7.90it/s]


WGAN-GP Epoch 2: c_loss=-2.1421, g_loss=0.9393, c_wasserstein=1.2660


WGAN-GP Epoch 3/100: 100%|██████████| 703/703 [01:29<00:00,  7.87it/s]


...

WGAN-GP Epoch 98: c_loss=-0.3789, g_loss=0.0625, c_wasserstein=1.8427


WGAN-GP Epoch 99/100: 100%|██████████| 703/703 [01:33<00:00,  7.50it/s]


WGAN-GP Epoch 99: c_loss=-0.3734, g_loss=0.1031, c_wasserstein=-5.5043


WGAN-GP Epoch 100/100: 100%|██████████| 703/703 [01:29<00:00,  7.88it/s]


WGAN-GP Epoch 100: c_loss=-0.3991, g_loss=0.1227, c_wasserstein=7.6561
Training completed. Performing final evaluation...
Final FID Score: 165.15
Final Inception Score: 1.47 ± 0.02


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import PathMNIST
from torch.utils.tensorboard import SummaryWriter

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
z_dim = 100
batch_size = 32
num_samples = 500  # Reduced to fit memory

# Data loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_dataset = PathMNIST(split='train', transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

# Generator class (must match your trained model)
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 3, 1, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# Load the trained generator
generator = Generator(z_dim=z_dim).to(device)
generator.load_state_dict(torch.load('best_generator_wgan_gp.pth'))
generator.eval()

# Generate fake images and add to tensorboard.
fake_images = []

with torch.no_grad():
    for _ in range(num_samples // batch_size):
        z = torch.randn(batch_size, z_dim, 1, 1).to(device)
        fake = generator(z)
        fake_images.append(fake.cpu())
fake_images = torch.cat(fake_images, dim=0)[:num_samples]

# Set up TensorBoard writer
writer = SummaryWriter('runs/wgan_generator_gp_samples')

# Add images to TensorBoard
writer.add_images('Generated Images', (fake_images + 1) / 2, 0) # normalize to 0-1 range.

#Add the model graph to tensorboard
z = torch.randn(1, z_dim, 1,1).to(device)
writer.add_graph(generator, z)

print("TensorBoard is ready. Run 'tensorboard --logdir=runs' in your terminal.")
writer.close()

Using device: cuda
Using downloaded and verified file: C:\Users\Sudhanshu\.medmnist\pathmnist.npz


C:\Users\Sudhanshu\AppData\Local\Temp\ipykernel_40144\1943896738.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  generator.load_state_dict(torch.load('best_generator_wg

TensorBoard is ready. Run 'tensorboard --logdir=runs' in your terminal.
